# Validación de Rotary Positional Embeddings (RoPE)

Este notebook valida la implementación aislada de RoPE antes de integrarla en el mecanismo de atención del modelo GPT.

Se comprobarán cuatro propiedades:

1. RoPE conserva las dimensiones del tensor.
2. La posición 0 no modifica el vector.
3. La rotación conserva la norma del vector.
4. El producto escalar entre Query y Key depende de su posición relativa.

In [4]:
import sys
from pathlib import Path

import torch

# Permitimos importar módulos desde ../src
project_root = Path.cwd().parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.rope import compute_rope_params, apply_rope

print("PyTorch version:", torch.__version__)

PyTorch version: 2.14.1+cpu


In [5]:
# Para reproducibilidad
torch.manual_seed(123)

BATCH_SIZE = 2
NUM_HEADS = 4
SEQ_LEN = 6
HEAD_DIM = 8

# Simulamos un tensor Query o Key
x = torch.randn(
    BATCH_SIZE,
    NUM_HEADS,
    SEQ_LEN,
    HEAD_DIM
)

print("Shape del tensor original:")
print(x.shape)

Shape del tensor original:
torch.Size([2, 4, 6, 8])


In [6]:
cos, sin = compute_rope_params(
    head_dim=HEAD_DIM,
    context_length=24
)

print("Shape cos:", cos.shape)
print("Shape sin:", sin.shape)

print("\ncos en posición 0:")
print(cos[0])

print("\nsin en posición 0:")
print(sin[0])

Shape cos: torch.Size([24, 8])
Shape sin: torch.Size([24, 8])

cos en posición 0:
tensor([1., 1., 1., 1., 1., 1., 1., 1.])

sin en posición 0:
tensor([0., 0., 0., 0., 0., 0., 0., 0.])


In [7]:
x_rope = apply_rope(
    x,
    cos,
    sin
)

print("Shape original:", x.shape)
print("Shape después de RoPE:", x_rope.shape)

Shape original: torch.Size([2, 4, 6, 8])
Shape después de RoPE: torch.Size([2, 4, 6, 8])


## Test 1 — Conservación de dimensiones

RoPE modifica los valores de Query y Key, pero no debe modificar las dimensiones del tensor.

Si la entrada tiene forma:

`(batch, heads, sequence_length, head_dimension)`

la salida debe mantener exactamente la misma forma.

In [8]:
assert x.shape == x_rope.shape, (
    f"Las dimensiones han cambiado: "
    f"{x.shape} -> {x_rope.shape}"
)

print("TEST 1 OK")
print("Shape antes :", x.shape)
print("Shape después:", x_rope.shape)

TEST 1 OK
Shape antes : torch.Size([2, 4, 6, 8])
Shape después: torch.Size([2, 4, 6, 8])


## Test 2 — La posición 0 no modifica el vector

Para la posición 0:

\[
\theta = 0
\]

por tanto:

\[
\cos(0)=1
\]

y:

\[
\sin(0)=0
\]

La transformación RoPE se convierte entonces en la identidad, por lo que el vector en posición 0 debe permanecer sin cambios.

In [9]:
original_position_0 = x[:, :, 0, :]
rope_position_0 = x_rope[:, :, 0, :]

same_position_zero = torch.allclose(
    original_position_0,
    rope_position_0,
    atol=1e-6,
    rtol=1e-6
)

print("¿Son iguales?", same_position_zero)

print("\nEjemplo original:")
print(original_position_0[0, 0])

print("\nEjemplo después de RoPE:")
print(rope_position_0[0, 0])

assert same_position_zero, (
    "RoPE está modificando incorrectamente la posición 0."
)

print("\nTEST 2 OK")

¿Son iguales? True

Ejemplo original:
tensor([ 0.3374, -0.1778, -0.3035, -0.5880,  0.3486,  0.6603, -0.2196, -0.3792])

Ejemplo después de RoPE:
tensor([ 0.3374, -0.1778, -0.3035, -0.5880,  0.3486,  0.6603, -0.2196, -0.3792])

TEST 2 OK


## Test 3 — Conservación de la norma

RoPE aplica una rotación a los vectores.

Una rotación puede modificar la dirección de un vector, pero no su longitud.

Por tanto debe cumplirse aproximadamente:

\[
||x|| = ||RoPE(x)||
\]

In [10]:
norm_before = torch.linalg.vector_norm(
    x,
    dim=-1
)

norm_after = torch.linalg.vector_norm(
    x_rope,
    dim=-1
)

same_norm = torch.allclose(
    norm_before,
    norm_after,
    atol=1e-5,
    rtol=1e-5
)

print("¿Se conserva la norma?", same_norm)

print("\nEjemplo de normas antes:")
print(norm_before[0, 0])

print("\nEjemplo de normas después:")
print(norm_after[0, 0])

max_norm_difference = (
    norm_before - norm_after
).abs().max()

print(
    "\nDiferencia máxima:",
    max_norm_difference.item()
)

assert same_norm, (
    "RoPE no está conservando correctamente la norma."
)

print("\nTEST 3 OK")

¿Se conserva la norma? True

Ejemplo de normas antes:
tensor([1.1545, 2.9030, 2.7887, 3.8580, 3.1467, 2.0114])

Ejemplo de normas después:
tensor([1.1545, 2.9030, 2.7887, 3.8580, 3.1467, 2.0114])

Diferencia máxima: 4.76837158203125e-07

TEST 3 OK


## Test 4 — Dependencia de la posición relativa

Una propiedad fundamental de RoPE es que la interacción entre Query y Key depende de la distancia relativa entre sus posiciones.

Compararemos:

- Query en posición 2 y Key en posición 5
- Query en posición 20 y Key en posición 23

En ambos casos la distancia relativa es:

\[
5 - 2 = 23 - 20 = 3
\]

Si utilizamos los mismos vectores Query y Key, sus productos escalares después de aplicar RoPE deberían ser aproximadamente iguales.

In [11]:
torch.manual_seed(456)

HEAD_DIM_RELATIVE = 8
CONTEXT_LENGTH = 24

# Creamos un único vector Query y un único vector Key
q = torch.randn(
    1,
    1,
    1,
    HEAD_DIM_RELATIVE
)

k = torch.randn(
    1,
    1,
    1,
    HEAD_DIM_RELATIVE
)

print("Query:")
print(q)

print("\nKey:")
print(k)

Query:
tensor([[[[-2.4306,  0.7544,  1.6025, -1.1431, -0.1645,  0.6943,  1.2980,
           -0.7403]]]])

Key:
tensor([[[[ 1.0322, -0.7191, -0.0373,  0.2765,  0.6191,  0.2177,  0.3027,
            0.3779]]]])


In [12]:
# Ponemos exactamente el mismo q y k
# en todas las posiciones de la secuencia

q_sequence = q.repeat(
    1,
    1,
    CONTEXT_LENGTH,
    1
)

k_sequence = k.repeat(
    1,
    1,
    CONTEXT_LENGTH,
    1
)

print("Shape Q:", q_sequence.shape)
print("Shape K:", k_sequence.shape)

Shape Q: torch.Size([1, 1, 24, 8])
Shape K: torch.Size([1, 1, 24, 8])


In [13]:
cos_relative, sin_relative = compute_rope_params(
    head_dim=HEAD_DIM_RELATIVE,
    context_length=CONTEXT_LENGTH
)

q_rotated = apply_rope(
    q_sequence,
    cos_relative,
    sin_relative
)

k_rotated = apply_rope(
    k_sequence,
    cos_relative,
    sin_relative
)

print("RoPE aplicado a Q y K.")

RoPE aplicado a Q y K.


In [14]:
# Caso 1:
# Query posición 2
# Key posición 5
# distancia = 3

score_2_5 = (
    q_rotated[:, :, 2, :]
    *
    k_rotated[:, :, 5, :]
).sum(dim=-1)


# Caso 2:
# Query posición 20
# Key posición 23
# distancia = 3

score_20_23 = (
    q_rotated[:, :, 20, :]
    *
    k_rotated[:, :, 23, :]
).sum(dim=-1)


print("Score posiciones 2 -> 5:")
print(score_2_5)

print("\nScore posiciones 20 -> 23:")
print(score_20_23)

Score posiciones 2 -> 5:
tensor([[1.9249]])

Score posiciones 20 -> 23:
tensor([[1.9249]])


In [15]:
same_relative_position = torch.allclose(
    score_2_5,
    score_20_23,
    atol=1e-5,
    rtol=1e-5
)

difference = (
    score_2_5 - score_20_23
).abs()

print(
    "¿Misma relación para igual distancia relativa?",
    same_relative_position
)

print(
    "Diferencia absoluta:",
    difference.item()
)

assert same_relative_position, (
    "No se conserva correctamente "
    "la relación de posición relativa."
)

print("\nTEST 4 OK")

¿Misma relación para igual distancia relativa? True
Diferencia absoluta: 1.1920928955078125e-07

TEST 4 OK


In [16]:
test_cases = [
    (1, 4),
    (2, 5),
    (5, 8),
    (10, 13),
    (20, 23),
]

scores = []

for q_pos, k_pos in test_cases:

    score = (
        q_rotated[:, :, q_pos, :]
        *
        k_rotated[:, :, k_pos, :]
    ).sum(dim=-1)

    scores.append(score.item())

    print(
        f"Q={q_pos:2d}, K={k_pos:2d}, "
        f"distancia={k_pos-q_pos:2d}, "
        f"score={score.item():.8f}"
    )

Q= 1, K= 4, distancia= 3, score=1.92486835
Q= 2, K= 5, distancia= 3, score=1.92486858
Q= 5, K= 8, distancia= 3, score=1.92486858
Q=10, K=13, distancia= 3, score=1.92486835
Q=20, K=23, distancia= 3, score=1.92486846


In [17]:
q_position = 2

key_positions = [
    3,   # distancia 1
    4,   # distancia 2
    5,   # distancia 3
    6,   # distancia 4
    7,   # distancia 5
]

print("Query fija en posición:", q_position)
print()

for k_position in key_positions:

    score = (
        q_rotated[:, :, q_position, :]
        *
        k_rotated[:, :, k_position, :]
    ).sum(dim=-1)

    distance = k_position - q_position

    print(
        f"Distancia={distance}, "
        f"score={score.item():.8f}"
    )

Query fija en posición: 2

Distancia=1, score=-1.01073003
Distancia=2, score=1.51200616
Distancia=3, score=1.92486858
Distancia=4, score=-0.20613253
Distancia=5, score=-2.97101974


In [18]:
print("=" * 50)
print("VALIDACIÓN RoPE")
print("=" * 50)

print("TEST 1 - Conservación dimensiones : OK")
print("TEST 2 - Posición cero             : OK")
print("TEST 3 - Conservación norma        : OK")
print("TEST 4 - Posición relativa         : OK")

print("=" * 50)
print("TODAS LAS PRUEBAS SUPERADAS")

VALIDACIÓN RoPE
TEST 1 - Conservación dimensiones : OK
TEST 2 - Posición cero             : OK
TEST 3 - Conservación norma        : OK
TEST 4 - Posición relativa         : OK
TODAS LAS PRUEBAS SUPERADAS
